# 학습코드

In [ ]:
# ════════════════════════════════════════════════════════════
# 4클래스 SSD 학습 — MobileNetV2 width_mult=0.5 백본 (PTQ 대상)
#
# 왜 V2인가 (V3-Small이 아니라):
#   V3-Small은 SE 블록 + Hardswish/Hardsigmoid가 있고, 이게 int8 양자화에서
#   무너져서(양자화 유지율 44%, 볼라드가 '사람'으로 오분류) 실패했다.
#   V2는 InvertedResidual + ReLU6 뿐이라 SE/Hardswish가 구조적으로 없다.
#   -> V3-Small이 무너진 바로 그 원인이 이 아키텍처에는 없다.
#
# ⭐ torchvision에 ssdlite_mobilenet_v2 팩토리가 없다.
#   V2의 InvertedResidual은 내부 시퀀셜을 `.conv`에 담는데(V3는 `.block`),
#   torchvision의 _mobilenet_extractor가 `.block`을 직접 참조해서 V2에는
#   그대로 못 쓴다 (AttributeError 실측 확인). 그래서 아래 V2Backbone을
#   직접 조립했다 — feats[:14](14x14, ch48) / feats[14:](7x7, ch1280) 두
#   단계로 나누고, SSDLite 스타일 depthwise-separable 다운샘플 블록
#   (extra_block) 4개를 이어붙여 4x4/2x2/1x1/1x1까지 만든다.
#   실측 검증 완료(로컬): 피처맵 [(14,14),(7,7),(4,4),(2,2),(1,1),(1,1)],
#   앵커 1602개(V3-Large/Small과 동일 — ssd_anchors.h 그대로 재사용 가능),
#   forward+backward+optimizer.step()까지 정상 동작 확인.
#   파라미터 1,081,028개 (V3-Small 1,556,996보다도 작음).
#
# ⭐ width_mult=0.5는 torchvision에 사전학습 체크포인트가 없다.
#   그래서 이번엔 V3-Small처럼 "일부만 사전학습"이 아니라 backbone+extra+head
#   전체가 랜덤 초기화다. 이게 의미하는 것:
#     - RGB->1채널 가중치 평균 전이가 필요 없다 (전이할 사전학습 가중치 자체가
#       없으므로). stem conv를 처음부터 1채널로 만든다.
#     - 차등 학습률(backbone 0.1x / head 1x) 구조가 더 이상 의미가 없다.
#       "사전학습된 걸 보호"할 대상이 없으므로 전체를 균일한 LR로 학습한다.
#     - V3-Small보다도 더 느리게 수렴할 수 있다. PATIENCE를 25->30으로
#       늘렸다. 그래도 부족하면 더 늘려야 할 수 있음 (실측 필요).
#
# ⭐ 데이터셋/증강은 기존 코드와 완전히 동일하게 유지했다 (일부러 안 바꿈).
#   바꾸면 "백본 때문인지 데이터 때문인지" 구분이 안 된다.
#   다만 알아둘 위험: 이번엔 모델 전체가 랜덤 초기화라, 자전거(1038장)처럼
#   원본이 적은 클래스를 WeightedRandomSampler가 반복 추출하면서 외워버릴
#   위험이 V3-Large/Small 때보다 크다. 사전학습된 특징이 없어서 기댈 게
#   없기 때문. 학습 후 자전거/킥보드 recall을 특히 확인할 것 — 유독 나쁘면
#   그때 증강을 보강(예: 약한 회전/이동/노이즈 추가)하는 게 맞는 순서다.


import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "tqdm", "-q"], check=False)

import os
import glob
import math
import random
import zipfile
import torch
import torch.nn as nn
import torch.utils.data as data
import numpy as np
from PIL import Image, ImageEnhance
from functools import partial
from collections import OrderedDict

from torchvision.models import mobilenet_v2
from torchvision.ops.misc import Conv2dNormActivation
from torchvision.models.detection.ssd import SSD
from torchvision.models.detection.ssdlite import SSDLiteHead
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform

# ════════════════════════════════════════════════════════════
# ── 설정 ─────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════
WIDTH_MULT = 0.5
IMG_SIZE = 224              # ⚠️ 바꾸면 앵커 개수가 달라져 ESP32 헤더/상수를 전부 고쳐야 함
EXPECTED_ANCHORS = 1602     # 224 기준 실측 검증 완료 (V3-Large/Small과 동일)

# ── 경로 ──
DRIVE_ROOT = '/content/drive/MyDrive/colab(new)'
base_out_dir = os.path.join(DRIVE_ROOT, '0815')   # 기존 0805v3, 0810_v3_small은 그대로 보존

# 드라이브가 안 붙었는데 여기 저장하면 makedirs가 로컬 임시 디스크에 그냥 만들어버린다.
# 몇 시간 학습하고 런타임이 끊기면 전부 사라진다 — 시작 전에 막는다.
if base_out_dir.startswith('/content/drive') and not os.path.ismount('/content/drive'):
    raise RuntimeError(
        "구글 드라이브가 마운트되지 않았습니다.\n"
        "  이 스크립트를 돌리기 전에 별도 셀에서 먼저 마운트하세요:\n"
        "    from google.colab import drive\n"
        "    drive.mount('/content/drive', force_remount=True)\n"
        "  안 붙은 채로 학습하면 체크포인트가 임시 디스크에 저장되어 런타임 종료 시 사라집니다.")

os.makedirs(base_out_dir, exist_ok=True)

DATASET_ZIP = os.path.join(DRIVE_ROOT, 'dataset_4class_final.zip')
extract_dir = '/content/object'

TRAIN_IMG_DIR = os.path.join(extract_dir, 'train', 'images')
TRAIN_LBL_DIR = os.path.join(extract_dir, 'train', 'labels')
VAL_IMG_DIR = os.path.join(extract_dir, 'val', 'images')
VAL_LBL_DIR = os.path.join(extract_dir, 'val', 'labels')

CLASS_NAMES = ['자전거', '킥보드', '볼라드', '사람']
NUM_CLASSES = len(CLASS_NAMES) + 1   # +1 = 배경(torchvision이 0번으로 예약)

# ── 하이퍼파라미터 ──
MAX_EPOCHS = 120
WARMUP_EPOCHS = 5
PATIENCE = 30             # 전체 랜덤 초기화라 V3-Small(25)보다 더 늦게 수렴할 수 있음 — 추정치, 실측 후 조정
BEST_TRACK_START = 20     # SSD 초기 "전부 배경" 예측 구간 함정 회피 (아키텍처 무관하게 발생)
BATCH_SIZE = 32
BASE_LR = 0.01            # 사전학습이 전혀 없어 차등 LR 불필요 -> 전체 균일 LR
NUM_WORKERS = 8

BEST_PATH = os.path.join(base_out_dir, 'best.pt')
LAST_PATH = os.path.join(base_out_dir, 'last.pt')

# ── 데이터셋 준비 ──
if not os.path.exists(os.path.join(extract_dir, 'data.yaml')):
    print("📦 데이터셋 압축 해제 중...")
    if not os.path.exists(DATASET_ZIP):
        raise FileNotFoundError(f"{DATASET_ZIP} 없음 - build_dataset_4class_final.py를 먼저 실행하세요")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(extract_dir)
    print("✅ 완료")
else:
    print("✅ 데이터셋 준비됨")


# ════════════════════════════════════════════════════════════
# 데이터셋 — 기존 코드와 완전히 동일 (일부러 안 바꿈, 위 설명 참고)
# ════════════════════════════════════════════════════════════
class YoloDataset(data.Dataset):
    """이미지가 이미 224 그레이스케일로 변환돼 있다는 전제로 최대한 가볍게 구성."""

    def __init__(self, img_dir, lbl_dir, augment=False):
        self.lbl_dir = lbl_dir
        self.augment = augment
        self.items = []
        self.classes_per_item = []

        for p in sorted(glob.glob(os.path.join(img_dir, '*.jpg'))):
            stem = os.path.splitext(os.path.basename(p))[0]
            lbl_path = os.path.join(lbl_dir, stem + '.txt')
            if not os.path.exists(lbl_path):
                continue

            boxes, classes_here = [], set()
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    c = int(parts[0])
                    boxes.append((c, *(float(x) for x in parts[1:])))
                    classes_here.add(c)
            if not boxes:
                continue

            self.items.append((p, boxes))
            self.classes_per_item.append(classes_here)

        print(f"[{img_dir}] {len(self.items)}장 (augment={augment})")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, boxes = self.items[idx]
        img = Image.open(img_path)

        flip = self.augment and random.random() < 0.5
        if flip:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if self.augment and random.random() < 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.25))
        if self.augment and random.random() < 0.5:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.75, 1.25))

        arr = np.asarray(img, dtype=np.float32) / 255.0
        if arr.ndim == 3:
            arr = arr[:, :, 0]
        img_t = torch.from_numpy(arr).unsqueeze(0)

        out_boxes, out_labels = [], []
        for c, cx, cy, w, h in boxes:
            if flip:
                cx = 1.0 - cx
            xmin = max(0.0, (cx - w / 2)) * IMG_SIZE
            ymin = max(0.0, (cy - h / 2)) * IMG_SIZE
            xmax = min(1.0, (cx + w / 2)) * IMG_SIZE
            ymax = min(1.0, (cy + h / 2)) * IMG_SIZE
            if xmax <= xmin or ymax <= ymin:
                continue
            out_boxes.append([xmin, ymin, xmax, ymax])
            out_labels.append(c + 1)

        if out_boxes:
            t_boxes = torch.tensor(out_boxes, dtype=torch.float32)
            t_labels = torch.tensor(out_labels, dtype=torch.int64)
        else:
            t_boxes = torch.zeros((0, 4), dtype=torch.float32)
            t_labels = torch.zeros((0,), dtype=torch.int64)

        return img_t, {"boxes": t_boxes, "labels": t_labels}


def collate_fn(batch):
    return tuple(zip(*batch))


def build_sampler(dataset):
    counts = [0] * len(CLASS_NAMES)
    for classes_here in dataset.classes_per_item:
        for c in classes_here:
            if 0 <= c < len(CLASS_NAMES):
                counts[c] += 1
    print(f"클래스별 등장 이미지 수: {dict(zip(CLASS_NAMES, counts))}")

    weights = []
    for classes_here in dataset.classes_per_item:
        valid = [counts[c] for c in classes_here if 0 <= c < len(CLASS_NAMES)]
        weights.append(1.0 / max(1, min(valid)) if valid else 1.0)
    return data.WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


# ════════════════════════════════════════════════════════════
# 모델 조립 — MobileNetV2(w=0.5) 커스텀 SSDLite
# ════════════════════════════════════════════════════════════
def _make_divisible(v, divisor=8):
    """torchvision의 채널 반올림 규칙과 동일 (8의 배수로)."""
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def _extra_block(in_ch, out_ch, norm_layer):
    """SSDLite 스타일 depthwise-separable 다운샘플 블록 (1x1 축소 -> depthwise 3x3 stride2 -> 1x1 확장).
    torchvision 내부 _extra_block과 동일한 설계, V2 의존성 없이 직접 구현."""
    mid = max(8, out_ch // 2)
    return nn.Sequential(
        Conv2dNormActivation(in_ch, mid, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, mid, kernel_size=3, stride=2, groups=mid,
                             norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, out_ch, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
    )


class V2Backbone(nn.Module):
    """MobileNetV2(width_mult) features를 2단계로 잘라 C1(14x14)/C2(7x7)로 쓰고,
    SSDLite 다운샘플 블록 4개를 이어붙여 4x4/2x2/1x1/1x1까지 만든다.
    (torchvision에 ssdlite_mobilenet_v2가 없어서 직접 조립 — 위 파일 상단 설명 참고)"""

    def __init__(self, width_mult, norm_layer):
        super().__init__()
        base = mobilenet_v2(weights=None, width_mult=width_mult, norm_layer=norm_layer)
        feats = base.features

        # 사전학습 가중치가 없으므로(width_mult=0.5는 torchvision에 체크포인트 없음)
        # RGB 평균 전이가 필요 없다 -> stem을 처음부터 1채널로 새로 만든다.
        stem_conv = feats[0][0]
        feats[0][0] = nn.Conv2d(1, stem_conv.out_channels, stem_conv.kernel_size,
                                stem_conv.stride, stem_conv.padding, bias=False)

        self.stage1 = nn.Sequential(*feats[:14])   # -> 14x14, ch=48(w=0.5 기준)
        self.stage2 = nn.Sequential(*feats[14:])   # -> 7x7,  ch=1280 (torchvision이 w<1.0에서도 고정)

        gd = lambda d: _make_divisible(d * width_mult)
        self.extra = nn.ModuleList([
            _extra_block(1280, gd(512), norm_layer),
            _extra_block(gd(512), gd(256), norm_layer),
            _extra_block(gd(256), gd(256), norm_layer),
            _extra_block(gd(256), gd(128), norm_layer),
        ])

    def forward(self, x):
        x = self.stage1(x)
        c1 = x
        x = self.stage2(x)
        c2 = x
        outs = [c1, c2]
        for block in self.extra:
            x = block(x)
            outs.append(x)
        return OrderedDict((str(i), v) for i, v in enumerate(outs))


def build_model():
    norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    backbone = V2Backbone(WIDTH_MULT, norm_layer)

    backbone.eval()
    with torch.no_grad():
        feats = backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
    out_channels = [f.shape[1] for f in feats.values()]

    anchor_generator = DefaultBoxGenerator([[2, 3] for _ in range(6)], min_ratio=0.2, max_ratio=0.95)
    model = SSD(
        backbone, anchor_generator, (IMG_SIZE, IMG_SIZE), NUM_CLASSES,
        head=SSDLiteHead(out_channels, anchor_generator.num_anchors_per_location(),
                         NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d),
        score_thresh=0.001, nms_thresh=0.55,
        detections_per_img=300, topk_candidates=300, positive_fraction=0.25,
    )
    model.transform = GeneralizedRCNNTransform(
        min_size=IMG_SIZE, max_size=IMG_SIZE, image_mean=[0.5], image_std=[0.5])

    # ── 앵커 개수 검증 ──
    model.eval()
    with torch.no_grad():
        fl = list(backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE)).values())
        n_anchor = model.head.classification_head(fl).shape[1]

    n_param = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*60}")
    print(f"모델: SSDLite + MobileNetV2(width={WIDTH_MULT})  @ {IMG_SIZE}x{IMG_SIZE}")
    print(f"  파라미터 : {n_param:,}  (V3-Small 1,556,996 / V3-Large 3,758,612 대비 참고)")
    print(f"  피처맵   : {[tuple(f.shape[-2:]) for f in fl]}")
    print(f"  채널     : {out_channels}")
    print(f"  앵커 개수: {n_anchor}  (기대값 {EXPECTED_ANCHORS})")
    print(f"  Hardsigmoid/Hardswish: "
          f"{sum(1 for m in model.modules() if isinstance(m, nn.Hardsigmoid))}/"
          f"{sum(1 for m in model.modules() if isinstance(m, nn.Hardswish))}  "
          f"(0/0이어야 정상 — SE·HardSwish 없음, V3-Small의 양자화 붕괴 원인이 구조적으로 없음)")
    if n_anchor != EXPECTED_ANCHORS:
        raise RuntimeError(
            f"앵커 개수가 {n_anchor}입니다 (기대 {EXPECTED_ANCHORS}).\n"
            f"ESP32 쪽 ssd_anchors.h를 dump_ssd_anchors.py로 재생성하고\n"
            f"model_inference_ssd.h의 SMARTCANE_SSD_NUM_ANCHORS도 {n_anchor}로 고쳐야 합니다.")
    print(f"  ✅ 앵커가 기존과 동일 -> ESP32는 model_data.cc만 교체하면 됩니다")
    print(f"{'='*60}\n")
    return model


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"디바이스: {device}")
    if device.type == 'cpu':
        print("⚠️ GPU가 아닙니다! 런타임 유형을 GPU로 바꿔주세요")

    train_ds = YoloDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR, augment=True)
    val_ds = YoloDataset(VAL_IMG_DIR, VAL_LBL_DIR, augment=False)

    train_loader = data.DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=build_sampler(train_ds),
        num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True,
        pin_memory=True, persistent_workers=True, prefetch_factor=4,
    )
    val_loader = data.DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True,
        pin_memory=True, persistent_workers=True,
    )

    model = build_model().to(device)

    # 사전학습이 전혀 없으므로(전체 랜덤 초기화) 차등 LR이 의미가 없다 -> 균일 LR 단일 그룹
    optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    start_epoch, best_val, no_improve = 0, float('inf'), 0
    if os.path.exists(LAST_PATH):
        ckpt = torch.load(LAST_PATH, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val = ckpt['best_val']
        no_improve = ckpt['no_improve']
        print(f"🔄 epoch {start_epoch}부터 이어서 학습 (best_val={best_val:.4f})")
    else:
        print("🆕 새로 학습 시작")

    from tqdm import tqdm

    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        total = 0.0
        bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}")
        for images, targets in bar:
            images = [i.to(device, non_blocking=True) for i in images]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total += loss.item()
            bar.set_postfix(loss=f"{loss.item():.3f}")

        scheduler.step()
        train_loss = total / max(1, len(train_loader))

        # SSD는 train() 모드에서만 loss를 반환하므로 train()은 유지하되, BN만 eval로 고정한다.
        # ⚠️ torch.no_grad()는 gradient만 막을 뿐 BatchNorm의 running_mean/var 갱신은
        #    그대로 일어난다. 안 하면 val_loss가 배치통계로 요동치고, val 데이터가
        #    BN 통계에 스며들어 export까지 따라간다.
        model.train()
        for mod in model.modules():
            if isinstance(mod, nn.BatchNorm2d):
                mod.eval()

        total_val = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images = [i.to(device, non_blocking=True) for i in images]
                targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
                total_val += sum(model(images, targets).values()).item()
        val_loss = total_val / max(1, len(val_loader))

        print(f"[{epoch+1}/{MAX_EPOCHS}] train={train_loss:.4f} val={val_loss:.4f} "
              f"lr={optimizer.param_groups[0]['lr']:.5f}")

        # SSD 초기엔 "전부 배경" 예측으로 val_loss가 인위적으로 낮다.
        # 그 구간 기록이 best로 굳으면 best.pt가 epoch 1짜리로 남는다 -> 여기서 리셋.
        if epoch + 1 == BEST_TRACK_START:
            print(f"  ↺ 초기 {BEST_TRACK_START}에폭 종료 - best 기준 리셋 "
                  f"(기존 best={best_val:.4f}는 '전부 배경' 예측 때문에 인위적으로 낮은 값)")
            best_val, no_improve = float('inf'), 0

        if val_loss < best_val:
            best_val, no_improve = val_loss, 0
            torch.save(model.state_dict(), BEST_PATH)
            print(f"  ✅ 최고 기록 갱신 (val={best_val:.4f})")
        else:
            no_improve += 1
            print(f"  개선 없음 ({no_improve}/{PATIENCE})")

        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'best_val': best_val, 'no_improve': no_improve}, LAST_PATH)

        if no_improve >= PATIENCE:
            print(f"\n🛑 {PATIENCE}에폭 연속 개선 없어 조기 종료")
            break

    print(f"\n✅ 완료. 최고 성능: {BEST_PATH} (val={best_val:.4f})")
    print("\n다음 단계:")
    print(f"  1) recall 측정 스크립트로 클래스별 recall 확인")
    print(f"     ⚠️ 특히 자전거/킥보드(원본이 적은 클래스) recall을 눈여겨볼 것 —")
    print(f"        전체 랜덤 초기화라 소수 클래스를 외워버렸을 위험이 V3-Small보다 큼")
    print(f"  2) int8 export (opset=14, hardsigmoid 관련 패치는 필요 없음 — 이 모델엔 없음)")
    print(f"     ⚠️ 그래도 export 직후 SUB 양자화 검사는 통과해야 함 (colab_2b 스크립트의")
    print(f"        check_sub_quant_safety 재사용 권장)")
    print(f"  3) ssd_anchors.h는 그대로 재사용 (앵커 {EXPECTED_ANCHORS}개 동일)")


if __name__ == '__main__':
    main()

📦 데이터셋 압축 해제 중...
✅ 완료
디바이스: cuda
[/content/object/train/images] 28764장 (augment=True)
[/content/object/val/images] 2534장 (augment=False)
클래스별 등장 이미지 수: {'자전거': 1038, '킥보드': 1416, '볼라드': 15000, '사람': 11310}

모델: SSDLite + MobileNetV2(width=0.5)  @ 224x224
  파라미터 : 1,081,028  (V3-Small 1,556,996 / V3-Large 3,758,612 대비 참고)
  피처맵   : [(14, 14), (7, 7), (4, 4), (2, 2), (1, 1), (1, 1)]
  채널     : [48, 1280, 256, 128, 128, 64]
  앵커 개수: 1602  (기대값 1602)
  Hardsigmoid/Hardswish: 0/0  (0/0이어야 정상 — SE·HardSwish 없음, V3-Small의 양자화 붕괴 원인이 구조적으로 없음)
  ✅ 앵커가 기존과 동일 -> ESP32는 model_data.cc만 교체하면 됩니다

🆕 새로 학습 시작


Epoch 1/120: 100%|██████████| 898/898 [02:15<00:00,  6.65it/s, loss=5.156]


[1/120] train=6.0122 val=5.2880 lr=0.00400
  ✅ 최고 기록 갱신 (val=5.2880)


Epoch 2/120: 100%|██████████| 898/898 [02:12<00:00,  6.79it/s, loss=4.002]


[2/120] train=4.6634 val=4.8061 lr=0.00600
  ✅ 최고 기록 갱신 (val=4.8061)


...

[118/120] train=0.7318 val=3.0377 lr=0.00001
  개선 없음 (25/30)


Epoch 119/120: 100%|██████████| 898/898 [02:08<00:00,  6.97it/s, loss=0.712]


[119/120] train=0.7338 val=3.0387 lr=0.00000
  개선 없음 (26/30)


Epoch 120/120: 100%|██████████| 898/898 [02:07<00:00,  7.02it/s, loss=0.603]


[120/120] train=0.7234 val=3.0368 lr=0.00000
  개선 없음 (27/30)

✅ 완료. 최고 성능: /content/drive/MyDrive/colab(new)/0815/best.pt (val=2.9901)

다음 단계:
  1) recall 측정 스크립트로 클래스별 recall 확인
     ⚠️ 특히 자전거/킥보드(원본이 적은 클래스) recall을 눈여겨볼 것 —
        전체 랜덤 초기화라 소수 클래스를 외워버렸을 위험이 V3-Small보다 큼
  2) int8 export (opset=14, hardsigmoid 관련 패치는 필요 없음 — 이 모델엔 없음)
     ⚠️ 그래도 export 직후 SUB 양자화 검사는 통과해야 함 (colab_2b 스크립트의
        check_sub_quant_safety 재사용 권장)
  3) ssd_anchors.h는 그대로 재사용 (앵커 1602개 동일)


# Recall

In [ ]:
# ════════════════════════════════════════════════════════════
# [1단계] 클래스별 recall 측정 — Large / Small / V2(w=0.5) 세 체크포인트를 한 번에 비교
#
# val_loss로는 판단할 수 없다: SSD의 loss는 classification+regression 합이고,
# 볼라드 15000 vs 자전거 1038의 14.5배 불균형에서는 다수 클래스가 loss를 지배한다.
# Large 대체 여부는 반드시 클래스별 recall로 판정할 것.
#
# 판정 기준(smartcane.md §6.5, Large @0805v3):
#   자전거 68.0 / 킥보드 85.9 / 볼라드 99.5 / 사람 58.1
#   -> 볼라드가 90% 아래로 떨어지면 속도를 얻은 대가가 너무 큼
#
# ⚠️ V2(w=0.5)는 전체가 랜덤 초기화로 학습됐다(사전학습 없음) — Small보다도
#    자전거/킥보드(원본 적은 클래스) recall이 나쁠 위험이 크다. 그 두 클래스를
#    특히 눈여겨볼 것.
# ════════════════════════════════════════════════════════════

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(드라이브 마운트 스킵: {e})")

import os, glob, zipfile
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from functools import partial
from collections import OrderedDict

from torchvision.models import mobilenet_v3_small, mobilenet_v2
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssd import SSD
from torchvision.models.detection.ssdlite import SSDLiteHead, _mobilenet_extractor
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection import _utils as det_utils
from torchvision.ops.misc import Conv2dNormActivation

# ── 설정 ──
DRIVE_ROOT = '/content/drive/MyDrive/colab(new)'

# (표시이름, 체크포인트, ARCH) — 없는 건 자동으로 건너뜀
CHECKPOINTS = [
    ('Large(0805v3)',  os.path.join(DRIVE_ROOT, '0805v3', 'best.pt'), 'v3_large'),
    ('Small(0810)',    os.path.join(DRIVE_ROOT, '0810_v3_small', 'best.pt'), 'v3_small'),
    ('V2w0.5(0815)',   os.path.join(DRIVE_ROOT, '0815', 'best.pt'), 'v2_w05'),
]

DATASET_ZIP = os.path.join(DRIVE_ROOT, 'dataset_4class_final.zip')
extract_dir = '/content/object'
VAL_IMG_DIR = os.path.join(extract_dir, 'val', 'images')
VAL_LBL_DIR = os.path.join(extract_dir, 'val', 'labels')

IMG_SIZE = 224
NUM_CLASSES = 5
CLASS_NAMES = ['자전거', '킥보드', '볼라드', '사람']
IOU_THRESHOLD = 0.5      # GT와 이만큼 겹치면 맞춘 것으로 인정
SCORE_THRESHOLD = 0.30   # ESP32의 kConfidenceThreshold와 동일하게

BASELINE = {'자전거': 68.0, '킥보드': 85.9, '볼라드': 99.5, '사람': 58.1}   # md §6.5
BOLLARD_MIN = 90.0        # 이 밑으로 떨어지면 그 체크포인트는 채택 불가

# ── 데이터셋 ──
if not os.path.exists(os.path.join(extract_dir, 'data.yaml')):
    print("📦 데이터셋 압축 해제 중...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(extract_dir)
    print("✅ 완료")


def patch_first_conv_to_1ch(root):
    """RGB 사전학습 가중치가 있는 백본(V3-Large/Small)용 — 평균 전이."""
    name, conv3 = None, None
    for n, m in root.named_modules():
        if isinstance(m, nn.Conv2d) and m.in_channels == 3:
            name, conv3 = n, m
            break
    conv1 = nn.Conv2d(1, conv3.out_channels, conv3.kernel_size, conv3.stride,
                      conv3.padding, bias=(conv3.bias is not None))
    parts, parent = name.split('.'), root
    for p in parts[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
    if parts[-1].isdigit():
        parent[int(parts[-1])] = conv1
    else:
        setattr(parent, parts[-1], conv1)


# ════════════════════════════════════════════════════════════
# V2(w=0.5) 커스텀 조립 — train_4class_v2_w05.py 와 완전히 동일해야
# state_dict가 로드된다 (구조가 1비트라도 다르면 load_state_dict가 실패한다)
# ════════════════════════════════════════════════════════════
V2_WIDTH_MULT = 0.5


def _make_divisible(v, divisor=8):
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def _extra_block(in_ch, out_ch, norm_layer):
    mid = max(8, out_ch // 2)
    return nn.Sequential(
        Conv2dNormActivation(in_ch, mid, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, mid, kernel_size=3, stride=2, groups=mid,
                             norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, out_ch, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
    )


class V2Backbone(nn.Module):
    def __init__(self, width_mult, norm_layer):
        super().__init__()
        base = mobilenet_v2(weights=None, width_mult=width_mult, norm_layer=norm_layer)
        feats = base.features
        stem_conv = feats[0][0]
        feats[0][0] = nn.Conv2d(1, stem_conv.out_channels, stem_conv.kernel_size,
                                stem_conv.stride, stem_conv.padding, bias=False)
        self.stage1 = nn.Sequential(*feats[:14])
        self.stage2 = nn.Sequential(*feats[14:])
        gd = lambda d: _make_divisible(d * width_mult)
        self.extra = nn.ModuleList([
            _extra_block(1280, gd(512), norm_layer),
            _extra_block(gd(512), gd(256), norm_layer),
            _extra_block(gd(256), gd(256), norm_layer),
            _extra_block(gd(256), gd(128), norm_layer),
        ])

    def forward(self, x):
        x = self.stage1(x); c1 = x
        x = self.stage2(x); c2 = x
        outs = [c1, c2]
        for block in self.extra:
            x = block(x); outs.append(x)
        return OrderedDict((str(i), v) for i, v in enumerate(outs))


def build_model(arch):
    """각 학습 스크립트의 build_model()과 정확히 동일한 구조여야 state_dict가 맞는다."""
    if arch == 'v2_w05':
        norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
        backbone = V2Backbone(V2_WIDTH_MULT, norm_layer)
        backbone.eval()
        with torch.no_grad():
            feats = backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
        out_ch = [f.shape[1] for f in feats.values()]
        ag = DefaultBoxGenerator([[2, 3] for _ in range(6)], min_ratio=0.2, max_ratio=0.95)
        model = SSD(backbone, ag, (IMG_SIZE, IMG_SIZE), NUM_CLASSES,
                    head=SSDLiteHead(out_ch, ag.num_anchors_per_location(),
                                     NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d),
                    score_thresh=0.001, nms_thresh=0.55,
                    detections_per_img=300, topk_candidates=300, positive_fraction=0.25)
        model.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                                   image_mean=[0.5], image_std=[0.5])
    elif arch == 'v3_small':
        norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
        bb = mobilenet_v3_small(weights=None, norm_layer=norm_layer, reduced_tail=False)
        backbone = _mobilenet_extractor(bb, 6, norm_layer)
        size = (IMG_SIZE, IMG_SIZE)
        ag = DefaultBoxGenerator([[2, 3] for _ in range(6)], min_ratio=0.2, max_ratio=0.95)
        out_ch = det_utils.retrieve_out_channels(backbone, size)   # 3채널 상태에서 먼저
        model = SSD(backbone, ag, size, NUM_CLASSES,
                    head=SSDLiteHead(out_ch, ag.num_anchors_per_location(),
                                     NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d),
                    score_thresh=0.001, nms_thresh=0.55,
                    detections_per_img=300, topk_candidates=300, positive_fraction=0.25)
        model.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                                   image_mean=[0.5], image_std=[0.5])
        patch_first_conv_to_1ch(model.backbone)                    # 그 다음 1채널로
    else:  # v3_large
        model = ssdlite320_mobilenet_v3_large(weights=None, weights_backbone=None)
        model.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                                   image_mean=[0.5], image_std=[0.5])
        patch_first_conv_to_1ch(model.backbone)
        model.backbone.eval()
        with torch.no_grad():
            feats = model.backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
        if isinstance(feats, torch.Tensor):
            feats = OrderedDict([("0", feats)])
        model.head = SSDLiteHead([f.size(1) for f in feats.values()],
                                 model.anchor_generator.num_anchors_per_location(),
                                 NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d)
    return model


def load_val():
    """(img_path, [(cls, x0,y0,x1,y1) 픽셀좌표]) 목록"""
    items = []
    for p in sorted(glob.glob(os.path.join(VAL_IMG_DIR, '*.jpg'))):
        lp = os.path.join(VAL_LBL_DIR, os.path.splitext(os.path.basename(p))[0] + '.txt')
        if not os.path.exists(lp):
            continue
        gts = []
        for line in open(lp):
            q = line.strip().split()
            if len(q) != 5:
                continue
            c, cx, cy, w, h = int(q[0]), *[float(x) for x in q[1:]]
            x0, y0 = max(0.0, cx - w/2) * IMG_SIZE, max(0.0, cy - h/2) * IMG_SIZE
            x1, y1 = min(1.0, cx + w/2) * IMG_SIZE, min(1.0, cy + h/2) * IMG_SIZE
            if x1 > x0 and y1 > y0:
                gts.append((c, x0, y0, x1, y1))
        if gts:
            items.append((p, gts))
    return items


def iou_matrix(gt, pred):
    """gt (N,4), pred (M,4) -> (N,M)"""
    if len(pred) == 0:
        return np.zeros((len(gt), 0))
    g = np.asarray(gt, dtype=np.float32)[:, None, :]
    p = np.asarray(pred, dtype=np.float32)[None, :, :]
    l = np.maximum(g[..., 0], p[..., 0]); t = np.maximum(g[..., 1], p[..., 1])
    r = np.minimum(g[..., 2], p[..., 2]); b = np.minimum(g[..., 3], p[..., 3])
    inter = np.clip(r - l, 0, None) * np.clip(b - t, 0, None)
    ga = (g[..., 2] - g[..., 0]) * (g[..., 3] - g[..., 1])
    pa = (p[..., 2] - p[..., 0]) * (p[..., 3] - p[..., 1])
    uni = ga + pa - inter
    return np.where(uni > 0, inter / np.maximum(uni, 1e-9), 0.0)


@torch.no_grad()
def evaluate(model, items, device):
    """클래스별 recall + 크기 구간별 recall"""
    model.eval().to(device)
    n_gt = np.zeros(4, dtype=np.int64)
    n_hit = np.zeros(4, dtype=np.int64)
    # 작은 객체를 놓치는지 따로 본다 (볼라드/원거리 대응 판단용)
    size_bins = [(0, 32), (32, 64), (64, 224)]
    b_gt = np.zeros(len(size_bins), dtype=np.int64)
    b_hit = np.zeros(len(size_bins), dtype=np.int64)
    n_pred_total = 0

    for i, (path, gts) in enumerate(items):
        g = np.array(Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
        out = model([torch.from_numpy(g).unsqueeze(0).to(device)])[0]

        keep = out['scores'].cpu().numpy() >= SCORE_THRESHOLD
        pb = out['boxes'].cpu().numpy()[keep]
        pl = out['labels'].cpu().numpy()[keep] - 1        # 0번은 배경
        n_pred_total += int(keep.sum())

        for c in range(4):
            gt_c = [(x0, y0, x1, y1) for cls, x0, y0, x1, y1 in gts if cls == c]
            if not gt_c:
                continue
            pr_c = pb[pl == c]
            n_gt[c] += len(gt_c)

            M = iou_matrix(gt_c, pr_c)
            hit = (M >= IOU_THRESHOLD).any(axis=1) if M.size else np.zeros(len(gt_c), bool)
            n_hit[c] += int(hit.sum())

            for k, (x0, y0, x1, y1) in enumerate(gt_c):
                side = max(x1 - x0, y1 - y0)
                for bi, (lo, hi) in enumerate(size_bins):
                    if lo <= side < hi:
                        b_gt[bi] += 1
                        b_hit[bi] += int(hit[k])
                        break

        if (i + 1) % 500 == 0:
            print(f"    {i+1}/{len(items)}...")

    return n_gt, n_hit, b_gt, b_hit, n_pred_total, size_bins


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
items = load_val()
print(f"\nval 이미지 {len(items)}장 / GT 박스 {sum(len(g) for _, g in items)}개")
print(f"판정 기준: IoU>={IOU_THRESHOLD}, score>={SCORE_THRESHOLD}\n")

results = {}
for name, ckpt, arch in CHECKPOINTS:
    if not os.path.exists(ckpt):
        print(f"⏭  {name}: {ckpt} 없음 - 건너뜀")
        continue
    print(f"▶ {name} 평가 중 ({arch})")
    model = build_model(arch)
    sd = torch.load(ckpt, map_location='cpu')
    if isinstance(sd, dict) and 'model' in sd:     # last.pt 형식도 허용
        sd = sd['model']
    model.load_state_dict(sd)
    results[name] = evaluate(model, items, device)
    print()

if not results:
    raise SystemExit("평가할 체크포인트가 없습니다")

# ── 결과 표 ──
print("=" * 90)
print(f"{'클래스':<8} {'GT':>6}", end='')
for name in results:
    print(f" {name:>16}", end='')
print(f" {'md 기록(Large)':>14}")
print("-" * 90)

for c in range(4):
    any_r = next(iter(results.values()))
    print(f"{CLASS_NAMES[c]:<8} {any_r[0][c]:>6}", end='')
    for name, (n_gt, n_hit, *_ ) in results.items():
        r = 100.0 * n_hit[c] / max(1, n_gt[c])
        print(f" {r:>15.1f}%", end='')
    print(f" {BASELINE[CLASS_NAMES[c]]:>13.1f}%")

print("-" * 90)
for name, (n_gt, n_hit, *_ ) in results.items():
    print(f"  {name}: 전체 recall {100.0*n_hit.sum()/max(1,n_gt.sum()):>5.1f}%  "
          f"(예측 박스 {results[name][4]}개)")

print("\n=== 객체 크기별 recall (긴 변 기준, 픽셀) ===")
for name, (_, _, b_gt, b_hit, _, bins) in results.items():
    print(f"  {name}")
    for bi, (lo, hi) in enumerate(bins):
        if b_gt[bi]:
            print(f"    {lo:>3}~{hi:<3}px : {100.0*b_hit[bi]/b_gt[bi]:>5.1f}%  (GT {b_gt[bi]}개)")

# ── 판정 (Large를 제외한 모든 후보 체크포인트에 동일 기준 적용) ──
print("\n=== 판정 ===")
for name, (n_gt, n_hit, *_ ) in results.items():
    if name.startswith('Large'):
        continue   # 기준 모델 자체는 판정 대상이 아님
    bollard = 100.0 * n_hit[2] / max(1, n_gt[2])
    bike = 100.0 * n_hit[0] / max(1, n_gt[0])
    kick = 100.0 * n_hit[1] / max(1, n_gt[1])
    if bollard < BOLLARD_MIN:
        print(f"  ❌ {name}: 볼라드 {bollard:.1f}% < {BOLLARD_MIN}% — 채택 불가")
    else:
        print(f"  ✅ {name}: 볼라드 {bollard:.1f}% 유지 — export 진행 가능")
    print(f"     자전거 {bike:.1f}% / 킥보드 {kick:.1f}%  "
          f"(Large 기준 68.0%/85.9% 대비 참고)")

print(f"\n※ 사람/자전거는 Large에서도 각각 58.1%/68.0%로 낮았던 클래스입니다.")
print(f"※ V2(w=0.5)는 사전학습이 전혀 없어(전체 랜덤 초기화) 자전거/킥보드처럼")
print(f"   원본이 적은 클래스를 특히 눈여겨봐야 합니다 — Small보다 더 나쁠 수 있습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

val 이미지 2534장 / GT 박스 5822개
판정 기준: IoU>=0.5, score>=0.3

▶ Large(0805v3) 평가 중 (v3_large)
    500/2534...
    1000/2534...
    1500/2534...
    2000/2534...
    2500/2534...

▶ Small(0810) 평가 중 (v3_small)
    500/2534...
    1000/2534...
    1500/2534...
    2000/2534...
    2500/2534...

▶ V2w0.5(0815) 평가 중 (v2_w05)
    500/2534...
    1000/2534...
    1500/2534...
    2000/2534...
    2500/2534...

클래스          GT    Large(0805v3)      Small(0810)     V2w0.5(0815)   md 기록(Large)
------------------------------------------------------------------------------------------
자전거         291            67.0%            53.6%            52.2%          68.0%
킥보드         170            84.7%            84.7%            84.1%          85.9%
볼라드         999            99.4%            97.4%            99.5%          99.5%
사람         4362            56.0%            48.7

In [ ]:
from google.colab import runtime

runtime.unassign()

# 양자화

In [ ]:
# ════════════════════════════════════════════════════════════
# Large / Small int8 export + 즉시 검증  (RELU_0_TO_1 재융합 대응판)
#
# ⚠️ 왜 고쳤나
#   기존 패치 `F.relu6(x+3)/6` 은 PyTorch 단계에서는 Hardsigmoid를 없애지만,
#   TFLite 변환기가 그 패턴을 다시 hard-sigmoid로 **재융합**해서 RELU_0_TO_1을
#   만들어낸다. TFLM에는 RELU_0_TO_1 커널이 아예 없어서(AddRelu/AddRelu6만 존재)
#   ESP32에서 못 돌린다.
#
#   그래서 패턴 매칭이 안 되는 형태로 바꾼다:
#       hardsigmoid(x) = (relu(x+3) - relu(x-3)) / 6      <- clip/relu6 형태가 아님
#       hardswish(x)   = x * (relu(x+3) - relu(x-3)) / 6
#   둘 다 RELU/SUB/MUL/ADD만 쓰므로 configure_ops()에 이미 전부 등록돼 있다.
#   (수학적 동치 확인: 최대오차 1.19e-07 = float32 반올림 수준)
#
# 자동 재시도: 먼저 Hardsigmoid만 분해해서 뽑고, 그래도 RELU_0_TO_1이 남으면
# Hardswish까지 분해해서 다시 뽑는다. Hardswish는 HARD_SWISH 전용 커널이 있어
# 가능하면 그대로 두는 게 빠르기 때문에 이 순서로 시도한다.
# ════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxsim", "onnx2tf",
                "onnx_graphsurgeon", "sng4onnx", "ai_edge_litert"], check=False)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(드라이브 마운트 스킵: {e})")

import os, glob, zipfile, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tensorflow as tf
from PIL import Image
from functools import partial
from collections import OrderedDict

from torchvision.models import mobilenet_v3_small, mobilenet_v2
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssd import SSD
from torchvision.models.detection.ssdlite import SSDLiteHead, _mobilenet_extractor
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection import _utils as det_utils
from torchvision.ops.misc import Conv2dNormActivation

# ── 설정 ──
DRIVE_ROOT = '/content/drive/MyDrive/colab(new)'
WIDTH_MULT_V2 = 0.5
TARGETS = [
    ('Small', 'v3_small', os.path.join(DRIVE_ROOT, '0810_v3_small', 'best.pt')),
    ('Large', 'v3_large', os.path.join(DRIVE_ROOT, '0805v3', 'best.pt')),
    ('V2w0.5', 'v2_w05', os.path.join(DRIVE_ROOT, '0815', 'best.pt')),
]
OUT_DIR = os.path.join(DRIVE_ROOT, 'Smartcane_export_compare')
WORK = '/content/export_cmp'

extract_dir = '/content/object'
DATASET_ZIP = os.path.join(DRIVE_ROOT, 'dataset_4class_final.zip')
REP_DIR = os.path.join(extract_dir, 'train', 'images')
REP_SAMPLES = 500

# 테스트 사진은 드라이브를 먼저 본다.
# /content 는 런타임이 끊기면 통째로 사라져서, 재시작할 때마다 다시 올려야 했다.
# 드라이브에 두면 유지된다. 위에 있는 경로부터 순서대로 찾는다.
TEST_IMAGE_DIRS = [
    os.path.join(DRIVE_ROOT, 'Smartcane'),   # 드라이브 (권장)
    '/content/test_images',                  # 로컬 업로드 (임시)
]
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

IMG_SIZE, NUM_CLASSES, EXPECT_ANCHORS = 224, 5, 1602
CLASS_NAMES = ['자전거', '킥보드', '볼라드', '사람']
CONF = 0.30

# model_inference_ssd.cc 의 configure_ops()가 등록한 연산자
TFLM_REGISTERED = {
    "CONV_2D", "DEPTHWISE_CONV_2D", "ADD", "MUL", "SUB", "CONCATENATION",
    "AVERAGE_POOL_2D", "MAX_POOL_2D", "FULLY_CONNECTED", "RESHAPE", "PAD", "PADV2",
    "LOGISTIC", "SOFTMAX", "QUANTIZE", "DEQUANTIZE", "TRANSPOSE_CONV", "TRANSPOSE",
    "SLICE", "HARD_SWISH", "RELU", "RELU6", "MEAN", "RESIZE_NEAREST_NEIGHBOR",
}

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

if not os.path.exists(os.path.join(extract_dir, 'data.yaml')):
    print("📦 데이터셋 압축 해제 중...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(extract_dir)

rep_files = sorted(glob.glob(os.path.join(REP_DIR, '*.jpg')))[:REP_SAMPLES]
print(f"대표데이터셋: {len(rep_files)}장 (train split)")

def collect_test_images():
    """후보 폴더를 순서대로 뒤져서 이미지만 모은다.
    ⚠️ '*.*' 로 긁으면 같은 폴더의 ssd_anchors.h 같은 파일까지 집으므로 확장자로 거른다."""
    print("테스트 사진 탐색:")
    found = []
    for d in TEST_IMAGE_DIRS:
        if not os.path.isdir(d):
            print(f"  ❌ {d}  (폴더 없음)")
            continue
        got = sorted(p for p in glob.glob(os.path.join(d, '*'))
                     if os.path.splitext(p)[1].lower() in IMG_EXTS)
        print(f"  {'✅' if got else '  '} {d}  → 이미지 {len(got)}장")
        for p in got:
            print(f"        {os.path.basename(p)}")
        found.extend(got)
        if got:
            break            # 먼저 찾은 폴더만 쓴다 (중복 방지)
    return found


test_imgs = collect_test_images()
if not test_imgs:
    raise SystemExit(
        "테스트 사진을 못 찾았습니다.\n"
        f"  다음 중 한 곳에 {'/'.join(IMG_EXTS)} 파일을 두세요:\n  - "
        + "\n  - ".join(TEST_IMAGE_DIRS) +
        "\n  드라이브 경로가 '폴더 없음'이면 마운트가 안 된 것입니다:\n"
        "    from google.colab import drive; drive.mount('/content/drive', force_remount=True)")
# ⚠️ 사진 한 장으로 비교하면 안 된다. 직전 실행에서 사진이 바뀌는 바람에
#    "Hardswish 분해 때문인지 사진 때문인지" 구분이 안 되는 상황이 생겼다.
#    또 Large는 float32 0.44 < int8 0.78 처럼 역전도 나온다 — 한 장은 그만큼 불안정하다.
TEST_SET = [(os.path.basename(p),
             np.array(Image.open(p).convert('L').resize((IMG_SIZE, IMG_SIZE)), dtype=np.uint8))
            for p in test_imgs]
print(f"테스트 사진 {len(TEST_SET)}장: {', '.join(n for n, _ in TEST_SET)}\n")


# ════════════════════════════════════════════════════════════
# 활성화 함수 분해 — 재융합되지 않는 형태
# ════════════════════════════════════════════════════════════
class HardSigmoidRelu6(nn.Module):
    """hardsigmoid(x) = relu6(x + 3) / 6

    ⭐ ESP32에서 정상 동작하던 기존 모델이 쓰던 형태다. 변환기가 이걸
       'conv bias에 +3 흡수' + 'conv fused activation = RELU6' + 'MUL(1/6)'
       으로 완전히 접어버려서, SE 블록이 CONV -> MUL -> MUL 로 깔끔해진다.
       (구 모델 실측: MEAN -> CONV -> CONV -> MUL(스칼라) -> MUL)

    ⚠️ relu 차분식 (relu(x+3)-relu(x-3))/6 은 쓰지 말 것.
       흡수되지 못하고 ADD+SUB+SUB가 그대로 남는데, 그 SUB의 출력 스케일이
       8e-9로 붕괴해서 real_output_multiplier가 2.86이 되고,
       TFLM의 QuantizeMultiplierSmallerThanOneExp가 abort한다. (실측 확인)
    """
    def forward(self, x):
        return F.relu6(x + 3.0) / 6.0


class HardSigmoidReluDiff(nn.Module):
    """(폴백용) relu 차분식. 위 경고대로 SUB 스케일 붕괴 위험이 있다."""
    def forward(self, x):
        return (F.relu(x + 3.0) - F.relu(x - 3.0)) / 6.0


class HardSwishReluDiff(nn.Module):
    """hardswish(x) = x * (relu(x+3) - relu(x-3)) / 6"""
    def forward(self, x):
        return x * ((F.relu(x + 3.0) - F.relu(x - 3.0)) / 6.0)


def patch_activations(mod, decompose_hardswish, hs_mode='relu6'):
    """hs_mode: 'relu6'(권장, 구 모델과 동일) | 'reludiff'(폴백)"""
    n_hs = n_hsw = 0
    for name, ch in mod.named_children():
        if isinstance(ch, nn.Hardsigmoid):
            setattr(mod, name,
                    HardSigmoidRelu6() if hs_mode == 'relu6' else HardSigmoidReluDiff())
            n_hs += 1
        elif decompose_hardswish and isinstance(ch, nn.Hardswish):
            setattr(mod, name, HardSwishReluDiff()); n_hsw += 1
        else:
            a, b = patch_activations(ch, decompose_hardswish, hs_mode)
            n_hs += a; n_hsw += b
    return n_hs, n_hsw


# ════════════════════════════════════════════════════════════
# 모델 조립 (학습 스크립트와 동일)
# ════════════════════════════════════════════════════════════
def patch_1ch(root):
    name, conv3 = None, None
    for n, m in root.named_modules():
        if isinstance(m, nn.Conv2d) and m.in_channels == 3:
            name, conv3 = n, m
            break
    c1 = nn.Conv2d(1, conv3.out_channels, conv3.kernel_size, conv3.stride,
                   conv3.padding, bias=(conv3.bias is not None))
    parts, par = name.split('.'), root
    for p in parts[:-1]:
        par = par[int(p)] if p.isdigit() else getattr(par, p)
    if parts[-1].isdigit():
        par[int(parts[-1])] = c1
    else:
        setattr(par, parts[-1], c1)


# ════════════════════════════════════════════════════════════
# V2(w=0.5) 백본 — train_4class_v2_w05.py 와 완전히 동일 (state_dict 호환 필수)
# ════════════════════════════════════════════════════════════
def _make_divisible(v, divisor=8):
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def _extra_block(in_ch, out_ch, norm_layer):
    mid = max(8, out_ch // 2)
    return nn.Sequential(
        Conv2dNormActivation(in_ch, mid, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, mid, kernel_size=3, stride=2, groups=mid,
                             norm_layer=norm_layer, activation_layer=nn.ReLU6),
        Conv2dNormActivation(mid, out_ch, kernel_size=1, norm_layer=norm_layer, activation_layer=nn.ReLU6),
    )


class V2Backbone(nn.Module):
    def __init__(self, width_mult, norm_layer):
        super().__init__()
        base = mobilenet_v2(weights=None, width_mult=width_mult, norm_layer=norm_layer)
        feats = base.features
        stem_conv = feats[0][0]
        feats[0][0] = nn.Conv2d(1, stem_conv.out_channels, stem_conv.kernel_size,
                                stem_conv.stride, stem_conv.padding, bias=False)
        self.stage1 = nn.Sequential(*feats[:14])
        self.stage2 = nn.Sequential(*feats[14:])
        gd = lambda d: _make_divisible(d * width_mult)
        self.extra = nn.ModuleList([
            _extra_block(1280, gd(512), norm_layer),
            _extra_block(gd(512), gd(256), norm_layer),
            _extra_block(gd(256), gd(256), norm_layer),
            _extra_block(gd(256), gd(128), norm_layer),
        ])

    def forward(self, x):
        x = self.stage1(x)
        c1 = x
        x = self.stage2(x)
        c2 = x
        outs = [c1, c2]
        for block in self.extra:
            x = block(x)
            outs.append(x)
        return OrderedDict((str(i), v) for i, v in enumerate(outs))


def build_model(arch):
    if arch == 'v2_w05':
        nl = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
        backbone = V2Backbone(WIDTH_MULT_V2, nl)
        backbone.eval()
        with torch.no_grad():
            feats = backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
        out_ch = [f.shape[1] for f in feats.values()]
        ag = DefaultBoxGenerator([[2, 3] for _ in range(6)], min_ratio=0.2, max_ratio=0.95)
        m = SSD(backbone, ag, (IMG_SIZE, IMG_SIZE), NUM_CLASSES,
                head=SSDLiteHead(out_ch, ag.num_anchors_per_location(),
                                 NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d),
                score_thresh=0.001, nms_thresh=0.55,
                detections_per_img=300, topk_candidates=300, positive_fraction=0.25)
        m.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                               image_mean=[0.5], image_std=[0.5])
        # ⚠️ V2Backbone은 stem을 처음부터 1채널로 만들기 때문에 patch_1ch()를 부르면 안 된다
        #    (부르면 이미 1채널인 conv를 또 찾다가 못 찾아 에러가 난다).
        return m
    elif arch == 'v3_small':
        nl = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
        bb = mobilenet_v3_small(weights=None, norm_layer=nl, reduced_tail=False)
        backbone = _mobilenet_extractor(bb, 6, nl)
        size = (IMG_SIZE, IMG_SIZE)
        ag = DefaultBoxGenerator([[2, 3] for _ in range(6)], min_ratio=0.2, max_ratio=0.95)
        out_ch = det_utils.retrieve_out_channels(backbone, size)
        m = SSD(backbone, ag, size, NUM_CLASSES,
                head=SSDLiteHead(out_ch, ag.num_anchors_per_location(),
                                 NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d),
                score_thresh=0.001, nms_thresh=0.55,
                detections_per_img=300, topk_candidates=300, positive_fraction=0.25)
        m.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                               image_mean=[0.5], image_std=[0.5])
        patch_1ch(m.backbone)
    else:
        m = ssdlite320_mobilenet_v3_large(weights=None, weights_backbone=None)
        m.transform = GeneralizedRCNNTransform(min_size=IMG_SIZE, max_size=IMG_SIZE,
                                               image_mean=[0.5], image_std=[0.5])
        patch_1ch(m.backbone)
        m.backbone.eval()
        with torch.no_grad():
            f = m.backbone(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
        if isinstance(f, torch.Tensor):
            f = OrderedDict([("0", f)])
        m.head = SSDLiteHead([t.size(1) for t in f.values()],
                             m.anchor_generator.num_anchors_per_location(),
                             NUM_CLASSES, norm_layer=torch.nn.BatchNorm2d)
    return m


class RawHead(nn.Module):
    """transform/postprocess는 빼고 backbone+head raw 출력만 내보낸다.

    ⚠️ 단, 정규화는 반드시 그래프 안에 넣어야 한다.
       학습 때 model(images, targets)는 GeneralizedRCNNTransform을 먼저 적용해서
       backbone에 (x-0.5)/0.5 = [-1,1] 을 준다. transform만 건너뛰고 [0,1]을 그대로
       먹이면 학습 조건과 다른 입력으로 추론하게 된다.
       (실측: 정규화하면 전체 모델 호출과 결과가 정확히 일치, 안 하면 어긋남)

       ESP32의 write_gray8_to_input()은 gray/255 = [0,1]을 보내므로,
       그래프가 [0,1]을 받아 안에서 정규화하는 이 형태가 맞다.
    """
    def __init__(self, ssd):
        super().__init__(); self.backbone = ssd.backbone; self.head = ssd.head
        # ⚠️ 정규화 상수를 파이썬 실수로 쓰면 안 된다.
        #    (x - 0.5) / 0.5 로 쓰면 나눗셈 상수가 tflite에서 shape=[] 인 rank-0 스칼라가
        #    되는데, TFLM의 MUL 커널이 rank-0 브로드캐스트를 못 다뤄 ESP32의
        #    AllocateTensors()에서 아무 로그도 없이 멈춘다. (실측으로 확인)
        #    정상 동작하던 기존 모델은 두 상수가 모두 shape=[1,1,1,1] 이었다.
        #    -> 형상을 가진 버퍼로 등록해서 [1,1,1,1] 상수가 되게 한다.
        self.register_buffer('norm_mean', torch.tensor(0.5).view(1, 1, 1, 1))
        self.register_buffer('norm_scale', torch.tensor(2.0).view(1, 1, 1, 1))

    def forward(self, x):
        x = (x - self.norm_mean) * self.norm_scale   # = (x - 0.5) / 0.5
        f = self.backbone(x)
        if isinstance(f, torch.Tensor):
            f = OrderedDict([("0", f)])
        fl = list(f.values())
        return self.head.regression_head(fl), self.head.classification_head(fl)


def representative_dataset():
    for p in rep_files:
        a = np.array(Image.open(p).convert('L').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
        yield [a.reshape(1, IMG_SIZE, IMG_SIZE, 1)]


def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def check_sub_quant_safety(path):
    """model_inference_ssd.cc가 링크하는 esp-tflite-micro의 sub_common.cc를 그대로
    재현해서, 모든 SUB 연산이 AllocateTensors()에서 abort 없이 통과하는지 미리 검사한다.
    (TFLITE_SCHEMA에서 int8 출력 기준 left_shift=20. 우리 그래프는 전부 int8이라 이걸로 충분)"""
    from tensorflow.lite.python import schema_py_generated as _schema
    m = _schema.ModelT.InitFromObj(_schema.Model.GetRootAsModel(bytearray(open(path, 'rb').read()), 0))
    sg = m.subgraphs[0]
    codes = [c.builtinCode if c.builtinCode else c.deprecatedBuiltinCode for c in m.operatorCodes]
    BOP_SUB = _schema.BuiltinOperator.SUB

    def scale_of(idx):
        q = sg.tensors[idx].quantization
        s = list(q.scale) if q and q.scale is not None else []
        return s[0] if s else None

    report = []
    for k, op in enumerate(sg.operators):
        if codes[op.opcodeIndex] != BOP_SUB:
            continue
        ins = [i for i in op.inputs if i >= 0]
        if len(ins) != 2:
            continue
        s1, s2, so = scale_of(ins[0]), scale_of(ins[1]), scale_of(op.outputs[0])
        if s1 is None or s2 is None or so is None or so == 0:
            continue
        twice_max = 2 * max(s1, s2)
        out_mult = twice_max / ((1 << 20) * so)
        in1_mult = s1 / twice_max
        in2_mult = s2 / twice_max
        ok = all(0.0 < v < 1.0 for v in (out_mult, in1_mult, in2_mult))
        report.append(dict(idx=k, out_mult=out_mult, ok=ok))
    return all(r['ok'] for r in report), report


def tflite_ops(path):
    it = tf.lite.Interpreter(model_path=path,
                             experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_REF)
    return sorted({d['op_name'] for d in it._get_ops_details()})


def eval_tflite(path, gray):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    inp = it.get_input_details()[0]; outs = it.get_output_details()
    cl = [o for o in outs if list(o['shape'])[-1] == NUM_CLASSES][0]
    if inp['dtype'].__name__ == 'int8':
        sc, zp = inp['quantization']
        x = np.clip(np.round(gray.astype(np.float32) / 255.0 / sc + zp), -128, 127).astype(np.int8)
    else:
        x = gray.astype(np.float32) / 255.0
    it.set_tensor(inp['index'], x.reshape(1, IMG_SIZE, IMG_SIZE, 1))
    it.invoke()
    c = it.get_tensor(cl['index'])[0]
    p = softmax(c)[:, 1:]
    ci = int(np.unravel_index(p.argmax(), p.shape)[1])
    return dict(top=float(p.max()), cls=CLASS_NAMES[ci], over=int((p >= CONF).sum()))


def eval_all(path):
    """테스트 사진 전부에 대해 평가 (한 장짜리 비교의 불안정성 회피)"""
    return {name: eval_tflite(path, g) for name, g in TEST_SET}


def export_once(arch, ckpt, decompose_hswish, tag, opset=14, hs_mode='relu6'):
    """한 번 export하고 (int8경로, float32경로, 연산자집합)을 돌려준다.

    ⚠️ opset이 결정적이다.
      opset 13 : ONNX에 HardSwish 연산자가 없어서 torch가 원시 연산으로 쪼갠다.
                 그 조각을 TFLite 변환기가 RELU_0_TO_1로 재융합해버린다(TFLM 미지원).
      opset 14+: ONNX에 HardSwish가 있어서 단일 노드로 나가고, TFLite HARD_SWISH로
                 매핑된다. 등록된 연산자인 데다 융합 커널이라 양자화도 훨씬 안정적이다.
                 (직접 분해하면 relu 차분에서 자리수 소실이 나서 int8이 무너진다)
    """
    model = build_model(arch)
    sd = torch.load(ckpt, map_location='cpu')
    if isinstance(sd, dict) and 'model' in sd:
        sd = sd['model']
    model.load_state_dict(sd)
    model.eval()

    n_hs, n_hsw = patch_activations(model, decompose_hswish, hs_mode)
    left_hs = sum(1 for m in model.modules() if isinstance(m, nn.Hardsigmoid))
    left_hsw = sum(1 for m in model.modules() if isinstance(m, nn.Hardswish))
    print(f"  활성화 분해: Hardsigmoid {n_hs}개({hs_mode}, 잔여 {left_hs}) / "
          f"Hardswish {n_hsw}개(잔여 {left_hsw}, 분해={decompose_hswish})")
    assert left_hs == 0

    w = RawHead(model).eval()
    with torch.no_grad():
        b, c = w(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
    assert c.shape[1] == EXPECT_ANCHORS, f"앵커 {c.shape[1]} != {EXPECT_ANCHORS}"

    onnx_p = os.path.join(WORK, f'{arch}_{tag}.onnx')
    sm_p = os.path.join(WORK, f'{arch}_{tag}_sm')
    if os.path.exists(sm_p):
        shutil.rmtree(sm_p)
    kw = dict(input_names=['input'], output_names=['bbox_regression', 'cls_logits'],
              opset_version=opset, do_constant_folding=True)
    try:
        torch.onnx.export(w, torch.zeros(1, 1, IMG_SIZE, IMG_SIZE), onnx_p, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(w, torch.zeros(1, 1, IMG_SIZE, IMG_SIZE), onnx_p, **kw)

    r = subprocess.run(['onnx2tf', '-i', onnx_p, '-o', sm_p, '-b', '1',
                        '--tflite_backend', 'tf_converter'], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError("onnx2tf 실패")

    f32_p = os.path.join(OUT_DIR, f'{arch}_float32.tflite')
    open(f32_p, 'wb').write(tf.lite.TFLiteConverter.from_saved_model(sm_p).convert())

    conv = tf.lite.TFLiteConverter.from_saved_model(sm_p)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = representative_dataset
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.float32     # ⚠️ 절대 int8 금지
    i8_p = os.path.join(OUT_DIR, f'{arch}_int8.tflite')
    open(i8_p, 'wb').write(conv.convert())

    # ⚠️ 정규화를 그래프에 넣으면 변환기가 그걸 입력 양자화 파라미터로 접어넣을 수 있다.
    #    그러면 그래프가 기대하는 입력 범위가 [0,1]이 아니라 [-1,1]이 되고,
    #    ESP32의 write_gray8_to_input()(gray/255 = [0,1] 전송)과 어긋난다.
    #    어느 쪽인지 반드시 눈으로 확인해야 한다.
    it = tf.lite.Interpreter(model_path=i8_p); it.allocate_tensors()
    ii = it.get_input_details()[0]
    sc, zp = ii['quantization']
    lo, hi = (-128 - zp) * sc, (127 - zp) * sc
    print(f"  입력 양자화: scale={sc:.8f} zp={zp} -> 그래프 기대 입력범위 [{lo:.3f}, {hi:.3f}]")
    if abs(lo) < 0.05 and abs(hi - 1.0) < 0.05:
        print("    ✅ [0,1] — ESP32의 write_gray8_to_input(gray/255) 그대로 맞습니다")
    elif abs(lo + 1.0) < 0.1 and abs(hi - 1.0) < 0.1:
        print("    ⚠️ [-1,1] — 변환기가 정규화를 입력 양자화로 접어넣었습니다.")
        print("       ESP32의 write_gray8_to_input()을 gray/255*2-1 로 고쳐야 합니다!")
    else:
        print(f"    ⚠️ 예상 밖의 범위 — ESP32 전처리와 대조가 필요합니다")

    # ⚠️ SUB 양자화 안전성 검사 — 이게 진짜 확인된 실패 조건이다.
    #
    # (rank-0 스칼라 상수는 문제 삼지 않는다. ESP32에서 실제로 정상 동작하던
    #  구 모델을 분석해보니 SE 블록 MUL에 rank-0 상수가 이미 1개 있었고 멀쩡히
    #  돌았다 — "rank-0이면 무조건 위험하다"는 판단은 틀렸다.)
    #
    # 실제로 확인된 사고는 특정 SUB 연산에서만 난다: TFLM의 CalculateOpDataSub()가
    #   real_output_multiplier = 2*max(in1_scale,in2_scale) / (2^20 * out_scale)
    # 를 계산해서 QuantizeMultiplierSmallerThanOneExp()에 넘기는데, 이 값이 (0,1)
    # 범위를 벗어나면 TFLITE_CHECK로 곧바로 abort한다. hardsigmoid를 relu 차분식
    # ((relu(x+3)-relu(x-3))/6)으로 분해했을 때 그 SUB의 출력 스케일이 8e-9까지
    # 붕괴해서 이 조건을 어겼던 게 실제로 겪은 사고였다 (백트레이스로 확인).
    ok_sub, sub_report = check_sub_quant_safety(i8_p)
    n_bad = sum(1 for r in sub_report if not r['ok'])
    print(f"    SUB 양자화 검사: {len(sub_report)}개 중 "
          f"{'전부 안전' if ok_sub else f'{n_bad}개 위험'}")
    if not ok_sub:
        for r in sub_report:
            if not r['ok']:
                print(f"      ❌ op[{r['idx']}] out_multiplier={r['out_mult']:.4f} "
                      f"(0~1 범위를 벗어남 → ESP32에서 abort)")
        raise RuntimeError(f"SUB 양자화 붕괴 {sum(1 for r in sub_report if not r['ok'])}건 "
                           f"— 이대로는 ESP32에서 못 씁니다")

    return i8_p, f32_p, set(tflite_ops(i8_p))


# ════════════════════════════════════════════════════════════
# 실행 — 자동 재시도
# ════════════════════════════════════════════════════════════
results = {}
for label, arch, ckpt in TARGETS:
    print("=" * 74)
    print(f"▶ {label} ({arch})")
    print("=" * 74)
    if not os.path.exists(ckpt):
        print(f"  ⏭ {ckpt} 없음 - 건너뜀\n"); continue

    # (opset, Hardswish 분해)  — 앞쪽이 더 좋은 조합이다.
    # HARD_SWISH를 융합 상태로 남기는 게 정확도·속도 모두에 유리하므로 opset 14/17을 먼저 본다.
    # 구 모델(ESP32 정상)과 같은 구조를 만드는 조합을 맨 앞에 둔다:
    #   opset14 -> HardSwish 유지, relu6형 hardsigmoid -> conv로 흡수
    ATTEMPTS = [(14, False, 'relu6'), (17, False, 'relu6'),
                (14, False, 'reludiff'), (13, True, 'relu6')]
    ok = False
    for attempt, (opset, dec, hsm) in enumerate(ATTEMPTS):
        tag = f'op{opset}_{"dec" if dec else "fused"}_{hsm}'
        print(f"  [시도 {attempt+1}] opset={opset}, Hardswish 분해={dec}, hardsigmoid={hsm}")
        try:
            i8_p, f32_p, ops = export_once(arch, ckpt, dec, tag, opset=opset, hs_mode=hsm)
        except Exception as e:
            print(f"     실패({type(e).__name__}): {str(e)[:150]} -> 다음 조합 시도\n")
            continue

        missing = sorted(ops - TFLM_REGISTERED)
        has_hswish = 'HARD_SWISH' in ops
        print(f"  연산자 {len(ops)}종 (HARD_SWISH 융합={'예' if has_hswish else '아니오'})")
        print(f"    {', '.join(sorted(ops))}")
        if missing:
            print(f"  ❌ TFLM 미등록: {missing} -> 다음 조합 시도\n")
            continue

        print(f"  ✅ 전부 TFLM 등록 연산자")
        results[label] = {'int8_path': i8_p, 'ops': ops,
                          'float32': eval_all(f32_p), 'int8': eval_all(i8_p),
                          'size': os.path.getsize(i8_p),
                          'hswish_decomposed': dec, 'opset': opset, 'fused': has_hswish,
                          'hs_mode': hsm}
        ok = True
        break
    if not ok:
        raise RuntimeError(f"{label}: 모든 조합이 실패했습니다")
    print()

# ── 결과 ──
print("=" * 84)
print("사진별 최고 전경확률  (float32 -> int8, 유지율)")
print("=" * 84)
for label, r in results.items():
    print(f"\n[{label}]  opset={r['opset']}  HARD_SWISH 융합={'예' if r['fused'] else '아니오'}  "
          f"크기 {r['size']:,} B")
    print(f"  {'사진':<22} {'float32':>10} {'int8':>10} {'유지율':>8} {'클래스(int8)':>12}")
    print("  " + "-" * 66)
    keeps = []
    for name, _ in TEST_SET:
        f, i = r['float32'][name], r['int8'][name]
        keep = i['top'] / f['top'] if f['top'] > 1e-9 else 0.0
        keeps.append(keep)
        print(f"  {name:<22} {f['top']:>10.4f} {i['top']:>10.4f} {keep*100:>7.0f}% {i['cls']:>12}")
    avg = sum(keeps) / len(keeps)
    verdict = "✅ 양자화 견딤" if avg > 0.7 else ("⚠️ 일부 손실" if avg > 0.3 else "❌ 붕괴")
    print(f"  {'평균 유지율':<22} {'':>10} {'':>10} {avg*100:>7.0f}%  {verdict}")

print("\n" + "=" * 84)
print("다음 단계: colab_3_make_model_data.py 를 아래 경로로 실행")
for label, r in results.items():
    print(f"  {label}: {r['int8_path']}")
print("\n※ 평균 유지율이 70% 아래면 ESP32에 올리지 마세요.")
print("※ 여기서 통과해도 클래스별 recall(1단계)은 별도로 반드시 측정해야 합니다.")


(드라이브 마운트 스킵: Error: credential propagation was unsuccessful)
📦 데이터셋 압축 해제 중...
대표데이터셋: 500장 (train split)
테스트 사진 탐색:
  ✅ /content/drive/MyDrive/colab(new)/Smartcane  → 이미지 4장
        볼라드.jpeg
        킥보드.jpeg
        킥보드2.jpeg
        킥보드3.jpeg
테스트 사진 4장: 볼라드.jpeg, 킥보드.jpeg, 킥보드2.jpeg, 킥보드3.jpeg

▶ Small (v3_small)
  [시도 1] opset=14, Hardswish 분해=False, hardsigmoid=relu6
  활성화 분해: Hardsigmoid 9개(relu6, 잔여 0) / Hardswish 0개(잔여 18, 분해=False)


/tmp/ipykernel_5291/3201758398.py:444: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(w, torch.zeros(1, 1, IMG_SIZE, IMG_SIZE), onnx_p, dynamo=False, **kw)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  입력 양자화: scale=0.00392157 zp=-128 -> 그래프 기대 입력범위 [0.000, 1.000]
    ✅ [0,1] — ESP32의 write_gray8_to_input(gray/255) 그대로 맞습니다
    SUB 양자화 검사: 1개 중 전부 안전
  연산자 12종 (HARD_SWISH 융합=예)
    ADD, CONCATENATION, CONV_2D, DEPTHWISE_CONV_2D, DEQUANTIZE, HARD_SWISH, MEAN, MUL, PAD, RESHAPE, SUB, TRANSPOSE
  ✅ 전부 TFLM 등록 연산자

▶ Large (v3_large)
  [시도 1] opset=14, Hardswish 분해=False, hardsigmoid=relu6
  활성화 분해: Hardsigmoid 8개(relu6, 잔여 0) / Hardswish 0개(잔여 20, 분해=False)


/tmp/ipykernel_5291/3201758398.py:444: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(w, torch.zeros(1, 1, IMG_SIZE, IMG_SIZE), onnx_p, dynamo=False, **kw)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  입력 양자화: scale=0.00392157 zp=-128 -> 그래프 기대 입력범위 [0.000, 1.000]
    ✅ [0,1] — ESP32의 write_gray8_to_input(gray/255) 그대로 맞습니다
    SUB 양자화 검사: 1개 중 전부 안전
  연산자 12종 (HARD_SWISH 융합=예)
    ADD, CONCATENATION, CONV_2D, DEPTHWISE_CONV_2D, DEQUANTIZE, HARD_SWISH, MEAN, MUL, PAD, RESHAPE, SUB, TRANSPOSE
  ✅ 전부 TFLM 등록 연산자

▶ V2w0.5 (v2_w05)
  [시도 1] opset=14, Hardswish 분해=False, hardsigmoid=relu6
  활성화 분해: Hardsigmoid 0개(relu6, 잔여 0) / Hardswish 0개(잔여 0, 분해=False)


/tmp/ipykernel_5291/3201758398.py:444: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(w, torch.zeros(1, 1, IMG_SIZE, IMG_SIZE), onnx_p, dynamo=False, **kw)


  입력 양자화: scale=0.00392157 zp=-128 -> 그래프 기대 입력범위 [0.000, 1.000]
    ✅ [0,1] — ESP32의 write_gray8_to_input(gray/255) 그대로 맞습니다
    SUB 양자화 검사: 1개 중 전부 안전
  연산자 10종 (HARD_SWISH 융합=아니오)
    ADD, CONCATENATION, CONV_2D, DEPTHWISE_CONV_2D, DEQUANTIZE, MUL, PAD, RESHAPE, SUB, TRANSPOSE
  ✅ 전부 TFLM 등록 연산자

사진별 최고 전경확률  (float32 -> int8, 유지율)

[Small]  opset=14  HARD_SWISH 융합=예  크기 2,033,448 B
  사진                        float32       int8      유지율    클래스(int8)
  ------------------------------------------------------------------
  볼라드.jpeg               0.7700     0.2055      27%           사람
  킥보드.jpeg               1.0000     0.2055      21%           사람
  킥보드2.jpeg              0.6225     0.4739      76%           사람
  킥보드3.jpeg              0.4081     0.2229      55%           사람
  평균 유지율                                            44%  ⚠️ 일부 손실

[Large]  opset=14  HARD_SWISH 융합=예  크기 2,862,584 B
  사진                        float32       int8      유지율    클래스(int8)
  --------

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


# model_data 뽑기


In [ ]:
# ════════════════════════════════════════════════════════════
# [3단계] .tflite -> ESP32용 model_data.cc / model_data.h
#
# 단순 변환만 하는 게 아니라, 쓰기 전에 ESP32 코드가 전제하는 조건을 전부 검사한다.
# 여기서 걸러내지 못하면 보드에 올린 뒤에야 "AllocateTensors failed"나
# "앵커 개수 불일치"로 알게 되고, 최악의 경우 조용히 틀린 값이 나온다.
#
# 특히 출력이 int8이면 무조건 막는다 — smartcane.md §5.3에 기록된 사고:
#   출력까지 int8로 강제했다가 bbox 양자화 범위가 -21.9~+4.82로 비대칭이 되어
#   볼라드 recall이 89.6% -> 42.7%로 붕괴했음.
# ════════════════════════════════════════════════════════════

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(드라이브 마운트 스킵: {e})")

import os
import numpy as np
import tensorflow as tf

# ── 설정 ──
DRIVE_ROOT = '/content/drive/MyDrive/colab(new)'
OUT_DIR    = '/content/model_data_out'

# ⭐ colab_2b_export_compare.py 가 만든 파일이 현재 정상 동작하는 버전이다.
#    (Smartcane_SSD_4class/ssd_4class_int8.tflite 는 대표데이터셋이 val 300장으로
#     잘못 뽑혀서 킥보드 최고확률이 0.0491까지 무너졌던 구버전 — 쓰면 안 됨)
TFLITE_PATH = os.path.join(DRIVE_ROOT, 'Smartcane_export_compare', 'v2_w05_int8.tflite')

# 위 경로가 없을 때만 순서대로 찾아본다. 어떤 파일을 쓰는지 항상 출력하므로
# 예전 파일을 모르고 쓰는 사고를 막는다.
CANDIDATES = [
    TFLITE_PATH,
    os.path.join(DRIVE_ROOT, 'Smartcane_export_compare', 'v3_small_int8.tflite'),
    os.path.join(DRIVE_ROOT, 'Smartcane_export_compare', 'v3_large_int8.tflite'),
    os.path.join(DRIVE_ROOT, 'Smartcane_SSD_4class', 'ssd_4class_int8.tflite'),
]

# ESP32 코드(model_inference_ssd.h)가 전제하는 값. 다르면 헤더/상수도 같이 고쳐야 한다.
EXPECT_INPUT_SHAPE = [1, 224, 224, 1]
EXPECT_NUM_CLASSES = 5      # 배경 + 4
EXPECT_NUM_ANCHORS = 1602
BYTES_PER_LINE     = 12     # 기존 model_data.cc와 동일한 서식

os.makedirs(OUT_DIR, exist_ok=True)

# ── 어떤 tflite를 쓸지 확정하고, 반드시 눈에 보이게 출력한다 ──
import time, datetime, hashlib
print("=== 입력 파일 선택 ===")
chosen = None
for c in CANDIDATES:
    if os.path.exists(c):
        mark = "◀ 사용" if chosen is None else ""
        age = (time.time() - os.path.getmtime(c)) / 60
        print(f"  {'✅' if chosen is None else '  '} {c}")
        print(f"       {os.path.getsize(c):,} B  "
              f"{datetime.datetime.fromtimestamp(os.path.getmtime(c)):%m-%d %H:%M} "
              f"({age:.0f}분 전) {mark}")
        if chosen is None:
            chosen = c
    else:
        print(f"  ❌ {c}  (없음)")

if chosen is None:
    raise FileNotFoundError("사용할 tflite가 없습니다 - 2단계 export를 먼저 실행하세요")
TFLITE_PATH = chosen

blob = open(TFLITE_PATH, 'rb').read()
print(f"\n입력: {TFLITE_PATH}")
print(f"  크기 {len(blob):,} B / 매직 {blob[4:8]} / md5 {hashlib.md5(blob).hexdigest()[:8]}")
if blob[4:8] != b'TFL3':
    raise RuntimeError("TFLite 파일이 아닙니다 (매직이 TFL3가 아님)")
if 'Smartcane_SSD_4class' in TFLITE_PATH:
    print("  ⚠️ 이 경로는 대표데이터셋이 잘못 뽑혔던 구버전일 수 있습니다.")
    print("     colab_2b_export_compare.py 를 먼저 돌려 새 파일을 만드는 것을 권합니다.")


# ════════════════════════════════════════════════════════════
# 검사 — ESP32에 올리기 전 마지막 관문
# ════════════════════════════════════════════════════════════
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
outs = interp.get_output_details()

print("\n=== ESP32 전제조건 검사 ===")
fatal = []


def check(label, ok, got, want, is_fatal=True):
    print(f"  [{'OK ' if ok else 'FAIL'}] {label:<32} got={got}  want={want}")
    if not ok and is_fatal:
        fatal.append(label)


check("입력 shape", list(inp['shape']) == EXPECT_INPUT_SHAPE,
      list(inp['shape']), EXPECT_INPUT_SHAPE)
check("입력 dtype (int8이어야 함)", inp['dtype'].__name__ == 'int8',
      inp['dtype'].__name__, 'int8')

# ⚠️ ESP32의 write_gray8_to_input()은 gray/255 = [0,1] 값을 보낸다.
# 그래프가 [-1,1]을 기대하면(정규화가 입력 양자화로 접혀 들어간 경우) 어긋나므로,
# 어느 쪽인지 판정해서 ESP32에서 뭘 해야 하는지 알려준다.
scale, zp = inp['quantization']
lo, hi = (-128 - zp) * scale, (127 - zp) * scale
is_01 = abs(lo) < 0.05 and abs(hi - 1.0) < 0.05
is_pm1 = abs(lo + 1.0) < 0.1 and abs(hi - 1.0) < 0.1
check("입력 역양자화 범위", is_01 or is_pm1, f"[{lo:.3f},{hi:.3f}]", "[0,1] 또는 [-1,1]")
if is_01:
    print("        ✅ [0,1] — ESP32 write_gray8_to_input(gray/255) 그대로 사용 가능")
elif is_pm1:
    print("        ⚠️ [-1,1] — ESP32 write_gray8_to_input()을 아래처럼 고쳐야 합니다:")
    print("           const float normalized = gray8[i] / 255.0f * 2.0f - 1.0f;")
    print("           (안 고치면 학습 조건과 다른 입력으로 추론하게 됩니다)")

check("출력 개수", len(outs) == 2, len(outs), 2)

bbox = [o for o in outs if list(o['shape'])[-1] == 4]
cls = [o for o in outs if list(o['shape'])[-1] == EXPECT_NUM_CLASSES]
check("bbox 출력(last_dim=4)", len(bbox) == 1, len(bbox), 1)
check("cls 출력(last_dim=5)", len(cls) == 1, len(cls), 1)

if bbox and cls:
    b, c = bbox[0], cls[0]
    # ⚠️ 여기가 제일 중요. 출력이 int8이면 절대 올리면 안 된다.
    check("bbox dtype (float32 필수)", b['dtype'].__name__ == 'float32',
          b['dtype'].__name__, 'float32')
    check("cls dtype (float32 필수)", c['dtype'].__name__ == 'float32',
          c['dtype'].__name__, 'float32')
    n_anchor = int(list(c['shape'])[1])
    check("앵커 개수", n_anchor == EXPECT_NUM_ANCHORS, n_anchor, EXPECT_NUM_ANCHORS)
    if n_anchor != EXPECT_NUM_ANCHORS:
        print(f"        -> ssd_anchors.h를 dump_ssd_anchors.py로 재생성하고")
        print(f"           model_inference_ssd.h의 SMARTCANE_SSD_NUM_ANCHORS를 {n_anchor}로 수정")

# TFLM resolver(configure_ops)가 등록한 연산자 목록
REGISTERED = {
    "CONV_2D", "DEPTHWISE_CONV_2D", "ADD", "MUL", "SUB", "CONCATENATION",
    "AVERAGE_POOL_2D", "MAX_POOL_2D", "FULLY_CONNECTED", "RESHAPE", "PAD", "PADV2",
    "LOGISTIC", "SOFTMAX", "QUANTIZE", "DEQUANTIZE", "TRANSPOSE_CONV", "TRANSPOSE",
    "SLICE", "HARD_SWISH", "RELU", "RELU6", "MEAN", "RESIZE_NEAREST_NEIGHBOR",
}
ref = tf.lite.Interpreter(model_path=TFLITE_PATH,
                          experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_REF)
ops = sorted({d['op_name'] for d in ref._get_ops_details()})
missing = [o for o in ops if o not in REGISTERED]
print(f"\n  사용 연산자 {len(ops)}종: {', '.join(ops)}")
if missing:
    print(f"  [FAIL] configure_ops()에 없는 연산자: {missing}")
    print(f"         -> model_inference_ssd.cc의 configure_ops()에 Add___() 를 추가해야 합니다")
    print(f"         -> RELU_0_TO_1 이 보이면 Hardsigmoid 패치가 빠진 것입니다")
    fatal.append("미등록 연산자")
else:
    print(f"  [OK ] 전부 configure_ops()에 등록돼 있음")

# ⚠️ SUB 양자화 안전성 검사.
# (rank-0 스칼라 상수는 그 자체로는 문제 없다 — ESP32에서 실제 정상 동작하던 구 모델도
#  SE 블록 MUL에 rank-0 상수가 1개 있었지만 멀쩡히 돌았다. "rank-0=위험"이라는 판단은 틀렸음.)
# 실제로 확인된 실패 조건은 특정 SUB 연산의 출력 양자화 배율이 (0,1) 범위를 벗어나는 것.
# TFLM의 CalculateOpDataSub()가 이 값을 TFLITE_CHECK로 검사해서 벗어나면 그 자리에서
# abort한다 (백트레이스로 직접 확인함). 그 수식을 그대로 재현해서 미리 검사한다.
try:
    from tensorflow.lite.python import schema_py_generated as _schema
    _m = _schema.ModelT.InitFromObj(_schema.Model.GetRootAsModel(bytearray(blob), 0))
    _sg = _m.subgraphs[0]
    _codes = [c.builtinCode if c.builtinCode else c.deprecatedBuiltinCode for c in _m.operatorCodes]

    def _scale_of(idx):
        q = _sg.tensors[idx].quantization
        s = list(q.scale) if q and q.scale is not None else []
        return s[0] if s else None

    _bad = []
    for _k, _op in enumerate(_sg.operators):
        if _codes[_op.opcodeIndex] != _schema.BuiltinOperator.SUB:
            continue
        _ins = [i for i in _op.inputs if i >= 0]
        if len(_ins) != 2:
            continue
        _s1, _s2, _so = _scale_of(_ins[0]), _scale_of(_ins[1]), _scale_of(_op.outputs[0])
        if _s1 is None or _s2 is None or _so is None or _so == 0:
            continue
        _tmax = 2 * max(_s1, _s2)
        _om = _tmax / ((1 << 20) * _so)
        _i1m, _i2m = _s1 / _tmax, _s2 / _tmax
        if not all(0.0 < v < 1.0 for v in (_om, _i1m, _i2m)):
            _bad.append((_k, _om))

    if _bad:
        print(f"\n  [FAIL] SUB 양자화 붕괴 {len(_bad)}건:")
        for _k, _om in _bad:
            print(f"         op[{_k}] out_multiplier={_om:.4f} (0~1 범위 벗어남 → ESP32에서 abort)")
        fatal.append("SUB 양자화 붕괴")
    else:
        print(f"\n  [OK ] SUB 양자화 안전 (전 SUB 연산 검사 완료)")
except Exception as e:
    print(f"\n  (SUB 양자화 검사 스킵: {e})")

if fatal:
    raise RuntimeError(f"검사 실패: {fatal}\n이 상태로 ESP32에 올리면 안 됩니다.")
print("\n✅ 모든 검사 통과\n")


# ════════════════════════════════════════════════════════════
# model_data.cc / .h 생성 (기존 파일과 동일한 서식)
# ════════════════════════════════════════════════════════════
cc_path = os.path.join(OUT_DIR, 'model_data.cc')
h_path = os.path.join(OUT_DIR, 'model_data.h')

with open(cc_path, 'w', newline='\n') as f:
    f.write('#include "model_data.h"\n\n')
    f.write('alignas(8) const unsigned char model_tflite[] = {\n')
    for i in range(0, len(blob), BYTES_PER_LINE):
        chunk = blob[i:i + BYTES_PER_LINE]
        f.write('  ' + ' '.join(f'0x{b:02x},' for b in chunk) + '\n')
    f.write('};\n')
    f.write(f'const unsigned int model_tflite_len = {len(blob)};\n')

with open(h_path, 'w', newline='\n') as f:
    f.write('#ifndef MODEL_DATA_H\n')
    f.write('#define MODEL_DATA_H\n\n')
    f.write('extern const unsigned char model_tflite[];\n')
    f.write('extern const unsigned int model_tflite_len;\n\n')
    f.write('#endif\n')

print(f"생성 완료:")
print(f"  {cc_path}  ({os.path.getsize(cc_path):,} B)")
print(f"  {h_path}   ({os.path.getsize(h_path):,} B)")

# 되읽어서 원본과 같은지 확인 (파싱 실수/서식 오류 방지)
import re
txt = open(cc_path, encoding='utf-8').read()
m = re.search(r"model_tflite\s*\[\s*\]\s*=\s*\{(.*?)\}\s*;", txt, re.S)
back = bytes(int(x, 16) for x in re.findall(r"0x([0-9a-fA-F]{2})", m.group(1)))
declared = int(re.search(r"model_tflite_len\s*=\s*(\d+)", txt).group(1))
print(f"\n왕복 검증: 되읽은 {len(back):,} B / 선언 {declared:,} / 원본과 동일 = {back == blob}")
if back != blob or declared != len(blob):
    raise RuntimeError("생성된 model_data.cc가 원본과 다릅니다")

print("\n다음 단계:")
print("  1) model_data.cc / model_data.h 를 다운로드")
print("  2) smartcane/main/ 의 같은 이름 파일을 덮어쓰기")
print("  3) ssd_anchors.h 는 그대로 (앵커 개수 동일)")
print("  4) VS Code에서 Build -> Flash -> Monitor")
print("  5) SMARTCANE_MEASURE_MODE=1 로 셀프테스트 추론시간 확인")

try:
    from google.colab import files
    files.download(cc_path)
    files.download(h_path)
except Exception:
    pass

(드라이브 마운트 스킵: Error: credential propagation was unsuccessful)
=== 입력 파일 선택 ===
  ✅ /content/drive/MyDrive/colab(new)/Smartcane_export_compare/v2_w05_int8.tflite
       1,517,160 B  08-14 23:15 (6분 전) ◀ 사용
     /content/drive/MyDrive/colab(new)/Smartcane_export_compare/v3_small_int8.tflite
       2,033,448 B  08-14 23:11 (10분 전) 
     /content/drive/MyDrive/colab(new)/Smartcane_export_compare/v3_large_int8.tflite
       2,862,584 B  08-14 23:13 (8분 전) 
     /content/drive/MyDrive/colab(new)/Smartcane_SSD_4class/ssd_4class_int8.tflite
       2,033,432 B  08-12 02:10 (4151분 전) 

입력: /content/drive/MyDrive/colab(new)/Smartcane_export_compare/v2_w05_int8.tflite
  크기 1,517,160 B / 매직 b'TFL3' / md5 75b71345

=== ESP32 전제조건 검사 ===
  [OK ] 입력 shape                         got=[np.int32(1), np.int32(224), np.int32(224), np.int32(1)]  want=[1, 224, 224, 1]
  [OK ] 입력 dtype (int8이어야 함)             got=int8  want=int8
  [OK ] 입력 역양자화 범위                       got=[0.000,1.000]  want=[0,1] 또는 [-1,1]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>